In [1]:
# Standard library imports
import os
from datetime import datetime

# Third-party imports
import numpy as np
import pandas as pd

# Workspace configuration
os.chdir("/work")
print(os.listdir("./pipeline"))

# Show in decimal rather than scientific notation
pd.options.display.float_format = '{:,.2f}'.format


['2.2.MUSICBRAINZ_mv_tmdb_soundtrack_wide_track_2015_2025.csv', '3.2.Artists_official_df.csv', '3.2.Wide_official_df.csv', '2.1.MUSICBRAINZ_mv_tmdb_soundtrack_album_spine_2015_2025.csv', '2.2.MUSICBRAINZ_mv_tmdb_soundtrack_album_artist_bridge_2015_2025.csv', '1.1.TMDB 2015-2025.csv', '2.2.MUSICBRAINZ_mv_tmdb_soundtrack_artist_spine_2015_2025.csv', '2.2.MUSICBRAINZ_mv_tmdb_soundtrack_track_spine_2015_2025.csv', '3.2.Tracks_official_df.csv', '3.2.Albums_official_df.csv', '3.3.Albums_deduped_df.csv', '3.3.Artists_deduped_df.csv', '3.3.Tracks_deduped_df.csv', '3.4.Albums_canonical_identified_df.csv', '3.3.Wide_deduped_df.csv', '3.4.Artists_canonical_identified_df.csv', '3.5.Albums_exploded_genre.csv', '3.4.Tracks_canonical_identified_df.csv', '3.4.Wide_canonical_identified_df.csv', '3.5.Artists_exploded_genre.csv', '3.5.Tracks_exploded_genre.csv', '3.5.Wide_exploded_genre.csv', '3.6.Albums_vote_count_analysis.csv', '3.6.Artists_vote_count_analysis.csv', '3.6.Tracks_vote_count_analysis.csv'

# QA of the Album and Film Table

This notebook focuses primarily on testing the integrity of the exported CSV files from MusicBrainz\. We look at the different attributes, assess their usefulness or appropriateness for future analysis, identify further data cleaning needs and just looking for any general weirdness in the data\.

In [2]:
# Load the albums dataframe

album_df = pd.read_csv("./pipeline/2.1.MUSICBRAINZ_mv_tmdb_soundtrack_album_spine_2015_2025.csv")
display(album_df.head())

# Display initial data state
display(album_df.columns)

,tmdb_id,film_title,film_adult,film_runtime_min,film_genres,film_rating,film_vote_count,film_mpaa_rating,film_original_title,film_language_name,...,release_tags_text,label_names,label_mbids,label_tags_text,rg_rating,rg_rating_count,release_meta_date_added,release_meta_info_url,release_meta_amazon_asin,release_meta_cover_art_presence
0,216015,Fifty Shades of Grey,False,125,"Drama, Romance, Thriller",5.88,12222,R,Fifty Shades of Grey,English,...,soundtrack,Republic Records | Universal Studios,9e4d8965-4fa4-4ce9-a48a-6db90769aa05 | e0ecd90...,NaN,NaN,NaN,2015-01-09 19:01:19.035 -0500,https://www.amazon.com/gp/product/B00S17FT82,NaN,present
1,216015,Fifty Shades of Grey,False,125,"Drama, Romance, Thriller",5.88,12222,R,Fifty Shades of Grey,English,...,NaN,Republic Records | Universal,1391bdc7-a22c-48a4-a5fb-e7b8ef6ce143 | e0ecd90...,alejandra guzman | clean up | es 2003 | sólo e...,NaN,NaN,2015-02-16 09:22:51.239 -0500,https://www.amazon.com/gp/product/B00S9303ZM,B00S9303ZM,present
2,150689,Cinderella,False,105,"Romance, Fantasy, Family, Drama",6.83,7294,PG,Cinderella,English,...,film score,Walt Disney Records,1c50034a-5c92-41a4-95ad-0acb7ea4686f,score | soundtrack | soundtrack label | classi...,NaN,NaN,2015-03-16 23:29:40.780 -0400,https://www.amazon.com/gp/product/B00THRE5TO,B00THRE5TO,present
3,281957,The Revenant,False,157,"Western, Drama, Adventure",7.54,18928,R,The Revenant,English,...,score | soundtrack,Milan,41099e29-7301-4fa3-86a4-8d510157b275,burbank | california | clean up | score | soun...,NaN,NaN,2015-12-31 15:04:24.681 -0500,https://www.amazon.com/gp/product/B018THTR7M,B018THTR7M,present
4,140607,Star Wars: The Force Awakens,False,136,"Adventure, Action, Science Fiction",7.30,20206,PG-13,Star Wars: The Force Awakens,English,...,soundtrack | classical | film score | star wars,Walt Disney Records,1c50034a-5c92-41a4-95ad-0acb7ea4686f,score | soundtrack | soundtrack label | classi...,93.00,3.00,2015-09-04 14:24:34.396 -0400,https://www.amazon.com/gp/product/B014V6JIQK,B014V6JIQK,present


Index(['tmdb_id', 'film_title', 'film_adult', 'film_runtime_min',
       'film_genres', 'film_rating', 'film_vote_count', 'film_mpaa_rating',
       'film_original_title', 'film_language_name', 'film_imdb_id',
       'film_wikidata_id', 'film_countries', 'film_year', 'film_release_date',
       'film_popularity', 'film_budget', 'film_revenue', 'film_studios',
       'film_director', 'film_soundtrack_composer_raw', 'film_top_cast',
       'film_keywords', 'film_ingested_at', 'match_method', 'soundtrack_type',
       'notes', 'matched_at', 'release_group_id', 'release_group_mbid',
       'album_title', 'rg_primary_type', 'rg_secondary_types', 'release_id',
       'release_mbid', 'release_title', 'release_status', 'release_packaging',
       'barcode', 'release_language', 'release_script', 'release_comment',
       'album_us_release_date', 'album_us_release_year',
       'album_us_release_month_min_observed',
       'album_us_release_day_min_observed', 'us_date_has_missing_month',
       

## A1\. Basic Data Integrity Checks

### Data Integrity Exploration A1\.1: Describe numeric columns

Question: Is there anything odd about the statistics or ranges around our numeric columns?

In [3]:
# Test A1.1: Numeric Column Analsysis

# 3. QA TEST A1.1: NUMERIC COLUMN ANALYSIS
# Properly indented hanging list
numeric_album_cols = [
    'film_vote_count', 
    'film_rating', 
    'film_popularity',
    'album_us_release_year', 
    'rg_rating', 
    'rg_rating_count'
]

print("\n--- Numeric Column Summary Statistics ---")
display(album_df[numeric_album_cols].describe())


--- Numeric Column Summary Statistics ---


,film_vote_count,film_rating,film_popularity,album_us_release_year,rg_rating,rg_rating_count
count,"5,209.00","5,209.00","5,209.00","1,510.00",274.00,274.00
mean,"1,302.79",6.40,3.48,"2,018.63",83.42,1.22
std,"2,965.81",0.90,10.16,2.58,20.24,0.67
min,10.00,2.50,0.00,"2,014.00",20.00,1.00
25%,36.00,5.84,0.77,"2,017.00",80.00,1.00
50%,179.00,6.48,1.81,"2,018.00",90.00,1.00
75%,"1,060.00",7.01,3.52,"2,021.00",100.00,1.00
max,"32,247.00",10.00,405.51,"2,025.00",100.00,7.00


Findings: Album\-level QA did not reveal material data quality issues; sparsity in U\.S\. release dates and MusicBrainz ratings is expected, and given that linkage confidence is already captured by match\_method, the numeric match\_score provides little additional analytical value\.

One thing is apparent, though: MusicBrainz ratings can likely be excluded from feature consideration\. Ratings cluster narrowly between 80 and 100, and rating counts are almost non\-existent, suggesting limited curator or community engagement with soundtrack rating on the platform\.

### Data Integrity Exploration A1\.2: ID completeness and uniqueness

Question: How complete are our ids?

In [4]:
# Test A1.2 ID completeness and uniqueness
id_columns = [
    'tmdb_id',
    'release_group_id', 
    'release_group_mbid', 
    'release_mbid', 
    'film_imdb_id', 
    'film_wikidata_id', 
    'barcode', 
    'label_mbids']

# Check for missing values in the existing ID columns
missing_values = album_df[id_columns].isna().sum()

# Display the count of missing values for each existing ID column
print(missing_values)
display(album_df[id_columns].head())

tmdb_id                  0
release_group_id         0
release_group_mbid       0
release_mbid             7
film_imdb_id            12
film_wikidata_id       399
barcode               2492
label_mbids            618
dtype: int64


,tmdb_id,release_group_id,release_group_mbid,release_mbid,film_imdb_id,film_wikidata_id,barcode,label_mbids
0,216015,1479583,85852629-ce4c-4bb8-8d4a-50299d56a06c,bdcaa988-1e80-446b-8e15-39b42255c4ed,tt2322441,Q14566553,"602,547,206,688.00",9e4d8965-4fa4-4ce9-a48a-6db90769aa05 | e0ecd90...
1,216015,1494030,fa7aa3a0-3de5-4273-bbb4-cfc13dbf9f1e,dd34d90a-52ff-445c-8221-d03a6f876b24,tt2322441,Q14566553,"602,547,201,720.00",1391bdc7-a22c-48a4-a5fb-e7b8ef6ce143 | e0ecd90...
2,150689,1504629,412fdd65-c3f8-40fe-a420-8b438e048915,5da9400f-7f0c-45f3-8e78-fe98f17aa43e,tt1661199,Q15046091,NaN,1c50034a-5c92-41a4-95ad-0acb7ea4686f
3,281957,1605141,5d5a40d3-8aab-41a8-8cfb-b97688a5c10c,bfff43a0-cac1-460a-94b9-fb10f70806c1,tt1663202,Q18002795,NaN,41099e29-7301-4fa3-86a4-8d510157b275
4,140607,1563177,405fd3c5-0a45-456a-b853-6f734d3b57aa,7f6687c0-daef-4a1a-aec5-b6edd1f3c7db,tt2488496,Q6074,"50,087,323,295.00",1c50034a-5c92-41a4-95ad-0acb7ea4686f


Findings: ID\-level QA indicates strong join integrity, with complete coverage for primary keys \(tmdb\_id, release\_group\_id, release\_group\_mbid\) and only expected sparsity across secondary identifiers such as IMDb/Wikidata IDs, barcodes, and label MBIDs \(which I don't think we will use anyway\)\.

In [5]:
dup_ct = album_df.duplicated(subset=["tmdb_id", "release_group_mbid"]).sum()
print("Duplicate (tmdb_id, release_group_mbid) rows:", dup_ct)


Duplicate (tmdb_id, release_group_mbid) rows: 0


Finding: Since \(tmdb\_id, release\_group\_mbid\) represents are composite primary key for this table, we need to make sure that there are indeed no duplicates\. Happy to say, there aren't any\!

### Data Integrity Exploration A1\.3: Categorical coverage and sparsity

Question: What are the different values in our categorical variables, and what is the categorical distribution?

In [6]:

# Test A1.3 Categorical coverage and sparsity
# This is more for interest -- I don't think we will be using these columns
categorical_album_cols = ['rg_primary_type', 'release_status', 'release_packaging', 'release_language', 'release_script']

# Check for unique values for those categories
for cat in categorical_album_cols:
    display(cat)
    display(album_df[cat].value_counts())

'rg_primary_type'

rg_primary_type
Album        4630
Single        288
EP            241
Other          10
Broadcast       1
Name: count, dtype: int64

'release_status'

release_status
Official          5036
Bootleg             53
Promotion           19
Withdrawn            7
Pseudo-Release       5
Name: count, dtype: int64

'release_packaging'

release_packaging
Jewel Case                196
Digipak                    47
Cardboard/Paper Sleeve     23
Other                      16
Gatefold Cover             13
Box                         9
Keep Case                   4
Discbox Slider              3
Digifile                    3
Cassette Case               3
Snap Case                   2
Super Jewel Box             2
Slim Jewel Case             2
Slidepack                   1
Digibook                    1
Clamshell Case              1
Name: count, dtype: int64

'release_language'

release_language
English                    3254
Hindi                       461
Tamil                       287
French                      130
Telugu                       99
[Multiple languages]         76
Japanese                     49
Malayalam                    33
Italian                      32
Spanish                      23
[No linguistic content]      16
Chinese                      11
Kannada                      11
Urdu                         10
German                        8
Korean                        7
Mandarin Chinese              6
Punjabi                       5
Filipino                      5
Dutch                         3
Marathi                       3
Portuguese                    3
Latin                         2
Arabic                        2
Turkish                       2
Georgian                      1
Norwegian                     1
Russian                       1
Persian                       1
Latvian                       1
Tagalog                

'release_script'

release_script
Latin                        4567
Japanese                       37
[Multiple scripts]             15
Telugu                         13
Han (Simplified variant)        8
Han (Traditional variant)       6
Korean                          6
Tamil                           4
Kannada                         3
Han (Hanzi, Kanji, Hanja)       1
Bengali                         1
Thai                            1
Arabic                          1
Cyrillic                        1
Name: count, dtype: int64

Finding: Categorical coverage appeared well\-structured, with long\-tailed but interpretable values reflecting MusicBrainz taxonomy and international release diversity\. Despite the U\.S\. film\-release scope, non\-English languages and non\-Latin scripts were not treated as deletion criteria and were retained as contextual metadata, with any potential filtering deferred to downstream measurement\-coverage QA\. 

By contrast, non\-Official release statuses \(e\.g\., Bootleg, Pseudo\-Release, Withdrawn\) should be excluded\. We will do this in a subsequent cleanup step \(Notebook 3\.2: Remove Unofficial Albums\) to ensure analytical consistency and avoid contaminating popularity and attribution metrics\.

### Data Integrity Exploration A1\.4: Label exploration

Question: How reliable is the label information?

In [7]:
label_cols = ["release_group_mbid", "album_title", "label_names", "label_mbids", "label_tags_text"]
labels_df = album_df.loc[:, label_cols].copy()

display(labels_df.isna().sum()/labels_df.shape[0])

release_group_mbid   0.00
album_title          0.00
label_names          0.12
label_mbids          0.12
label_tags_text      0.44
dtype: float64

Finding: Only 12% of albums have null for the label, making them pretty reliable

In [8]:
display(labels_df['label_names'].value_counts())
display(labels_df['label_names'].value_counts().head(25))

label_names
Lakeshore Records         331
Milan                     244
Back Lot Music            206
T‐Series                  194
Sony Music                170
                         ... 
Arts & Crafts               1
Doxy                        1
EastWest                    1
TOHO animation RECORDS      1
SVF Music                   1
Name: count, Length: 939, dtype: int64

label_names
Lakeshore Records           331
Milan                       244
Back Lot Music              206
T‐Series                    194
Sony Music                  170
[no label]                  159
Zee Music Company           156
WaterTower Music            133
Walt Disney Records         131
Sony Classical              121
Hollywood Records            99
Netflix Music                96
Varèse Sarabande             92
Think Music                  76
MovieScore Media             64
Saregama                     42
Filmtrax                     41
Decca Records                40
Madison Gate Records         40
A24 Music                    37
Quartet Records              36
Masterworks                  36
Aditya Music                 36
Lahari Music                 27
Plaza Mayor Company Ltd.     25
Name: count, dtype: int64

Finding: Label metadata is well populated and structurally consistent, with clear concentration among a small number of soundtrack\-focused and major music labels \(e\.g\., Lakeshore Records, Milan, Back Lot Music, Sony\)\. However, the distribution is extremely long\-tailed, with nearly 1,000 distinct labels and the majority appearing only once\. In addition, label identity reflects business and distribution relationships rather than musical characteristics\. As a result, we will probably keep label data as descriptive context, and perhaps do some nice top labels analysis or visualizations, but we will not treat it as a primary analytical feature\.

In [9]:
# %% TMDB composer coverage (album_df)

tmdb_has_composer = (
    album_df["film_soundtrack_composer_raw"].notna()
    & (album_df["film_soundtrack_composer_raw"].astype(str).str.strip() != "")
)

tmdb_film_coverage = (
    album_df.loc[:, ["tmdb_id"]]
       .assign(tmdb_has_composer=tmdb_has_composer)
       .groupby("tmdb_id")["tmdb_has_composer"]
       .any()
)

pd.Series({
    "films_total": int(tmdb_film_coverage.shape[0]),
    "films_with_tmbd_composer": int(tmdb_film_coverage.sum()),
    "pct_films_with_tmdb_composer": float(tmdb_film_coverage.mean() * 100),
})


films_total                    4,776.00
films_with_tmbd_composer       4,776.00
pct_films_with_tmdb_composer     100.00
dtype: float64

Findings: Composer attribution from TMDB is complete at the film level\. All films in the album dataset have populated soundtrack composer information in TMDB, yielding 100% coverage\. This contrasts with the sparse and inconsistent composer metadata observed in MusicBrainz and indicates that TMDB provides a more reliable and structurally aligned source for film\-level composer analysis\.

## A2\. Trickier Data Integrity Checks

### Data Integrity Exploration A2\.1: Inspect films with multiple albums

In [10]:
# Test A2.1 Inspect films with multiple albums
multi_films = album_df.groupby('tmdb_id').size()

# Identify films with more than 1 album
multi_films = multi_films[multi_films > 1].index

# display(len(multi_albums))   # 494 multi-albums
cols = [
    "tmdb_id", "film_title", "film_year","film_soundtrack_composer_raw",
    "album_title", "album_us_release_year","rg_secondary_types",
    "soundtrack_type", "match_method", 
    "release_group_mbid", "release_mbid"
]

# Identify rows which are in our multi_films list. Show most useful columns for inspection
multi_album_df = album_df.loc[album_df['tmdb_id'].isin(multi_films), cols]. \
    sort_values(['tmdb_id', 'film_title', 'album_us_release_year', 'album_title'])

display(multi_album_df)

,tmdb_id,film_title,film_year,film_soundtrack_composer_raw,album_title,album_us_release_year,rg_secondary_types,soundtrack_type,match_method,release_group_mbid,release_mbid
1686,38700,Bad Boys for Life,2020,Lorne Balfe,Bad Boys for Life,"2,020.00",Soundtrack,unknown,title_contains_strict,59cb736a-add2-4de5-b999-8f8b4fd88522,059bd09c-f605-40f5-8906-88692cf74b2a
1496,38700,Bad Boys for Life,2020,Lorne Balfe,Bad Boys for Life: The Soundtrack,NaN,Soundtrack,songs,title_contains_strict,155b9ac1-1b80-4aaa-820d-c6d7bdf9ff97,707dc0ed-3ce0-4d3b-b15c-b806482bb30c
141,43074,Ghostbusters,2016,Theodore Shapiro,Ghostbusters (Original Motion Picture Score),"2,016.00",Soundtrack,score,imdb_exact,330c9947-62fc-483a-ba3e-2ef2cdba1322,8a9fd4cc-79d4-4e34-9a1a-33ce4534315c
140,43074,Ghostbusters,2016,Theodore Shapiro,Ghostbusters: Original Motion Picture Soundtrack,"2,016.00",Soundtrack,songs,imdb_exact,6cd14cca-1d16-4954-8f78-8e25b932c0b5,6326805b-927e-4da2-b853-e9b2a189efd2
4437,47971,xXx: Return of Xander Cage,2017,"Brian Tyler, Robert Lydecker",xXx: Return of Xander Cage,"2,017.00",Soundtrack,unknown,title_contains_strict,55976e08-6daa-4da6-bd3b-a5539e2b7915,834aa4a3-6f15-4aaf-ab05-0f2a4138a381
...,...,...,...,...,...,...,...,...,...,...,...
2006,1293244,Four Letters of Love,2025,Anne Nikitin,Four Letters of Love: Original Film Soundtrack,NaN,Soundtrack,songs,title_contains_strict,078c7351-00e2-4483-9996-3d7a0c24242d,860b702f-0b7f-45e9-9f97-f07655d719f8
3811,1368166,The Housemaid,2025,Theodore Shapiro,"The Housemaid, Vol. 1: Original Motion Picture...",NaN,Soundtrack,songs,title_contains_strict,bf5aec28-3597-4fae-847f-03d372690b5c,30e81a52-263f-4017-98fe-bf57dbe42f12
3945,1368166,The Housemaid,2025,Theodore Shapiro,"The Housemaid, Vol. 2: Original Motion Picture...",NaN,Soundtrack,songs,title_contains_strict,c75581c3-f1ea-412b-9dc6-306345b5ba73,4faf6d2d-79a7-41b0-8c72-f43f985326a3
4666,1380998,The Book,2024,Unknown,THE BOOK OF CLARENCE: Original Motion Picture ...,NaN,Soundtrack,songs,title_contains_strict,ad59e0d7-5ad1-4531-a9a8-d7f5a2f9a695,ed57f9a7-66fa-4af1-aafa-c39159b8d146


Findings: Inspecting films with multiple associated albums shows that “multi\-album” cases are usually explainable rather than obviously erroneous: many films have both a score release and a songs/soundtrack release \(e\.g\., Ghostbusters, Furious 7, Black Panther\), and some also include single\-style releases \(“Theme From…”, “From the Motion Picture…”\) that appear as additional album rows\. 

A smaller subset of multi\-album rows are essentially duplicate/ambiguous album titles \(often soundtrack\_type = unknown and/or match\_method = title\_contains\_strict\), which look like weaker title\-based associations compared to the imdb\_exact matches\. Overall, the output suggests that canonically selecting one album per film will require a clear prioritization rule \(e\.g\., prefer score vs songs, downweight single\-like and unknown/title\-only matches\) rather than treating all multi\-album films as data quality issues\.

### Data Integrity Exploration A2\.2: Test Film to album temporal logic

In [11]:

# Test A2.2 Test Film to album temporal logic
album_temporal_df = album_df[['film_title', 'film_year', 'album_title', 'album_us_release_year', 
'tmdb_id', 'release_group_mbid', 'release_mbid', 'release_id', 'release_group_id']].copy()

# From the previous test, we know that only 1590 out of the 5328 albums have us_release_dates
album_temporal_df['album_us_release_year'].isna().sum()
# 3738 albums have no U.S. release date

# Test A2.2.1: Are all albums released after the film?
album_temporal_df['album_minus_film_release'] = album_temporal_df['album_us_release_year'] - album_temporal_df['film_year']
album_temporal_df['album_minus_film_release'].value_counts()

display(album_temporal_df.sort_values(by='album_minus_film_release', ascending=True).head(50))



,film_title,film_year,album_title,album_us_release_year,tmdb_id,release_group_mbid,release_mbid,release_id,release_group_id,album_minus_film_release
2924,All Summers End,2017,All Summers End: Original Score,"2,014.00",322517,ed696894-074c-4231-af1f-3fb664025d43,02fbca2a-656e-474a-a11f-147d65d3471b,"3,171,520.00",2756295,-3.00
3654,The Nest,2021,The Nest (Original Motion Picture Soundtrack),"2,020.00",790803,66b79a4b-3295-42d2-b002-56155a2e4dcd,d4f9df15-caed-4c2d-8acb-9b397cebb90c,"2,868,563.00",2515809,-1.00
3911,Play,2019,Play,"2,018.00",629354,e56ebed6-a10b-4f58-b7c1-ea4891717eb2,15275a53-7d71-4202-8360-7e0da1668b6d,"2,230,542.00",2015647,-1.00
3912,Heaven,2019,Heaven,"2,018.00",532024,669f4478-f41b-4345-ae38-276097159925,bd916c23-20e6-41c1-87ff-488620c50bb8,"2,109,420.00",1918776,-1.00
2070,The Other,2025,The Other,"2,024.00",1162858,76248ff3-b62b-411d-8d6d-71d987712c5d,05ec1a8d-d9d6-47ed-a80c-beef83eb6388,"1,089,493.00",1107149,-1.00
3920,Godzilla,2015,Godzilla: Original Motion Picture Soundtrack,"2,014.00",1125950,1525cf28-f2e5-4af0-b872-72af7433cd30,8e3ce838-b048-4aea-9042-0fc522bd1a20,"1,425,543.00",1375213,-1.00
4359,Gold,2018,Gold,"2,017.00",478682,a310dd89-20f0-4252-9ef6-73c8614b9782,ffc3ff4c-c744-47c7-9196-9ac48649a4b1,"2,088,482.00",1902647,-1.00
1688,The Book Of Vision,2021,The Book of Vision,"2,020.00",498587,597d1b00-cf7c-47fb-ab57-9545c764b0f3,46765523-90d5-4cb4-929c-650c33d1b9dc,"2,805,183.00",2467424,-1.00
3937,Away,2025,Away,"2,024.00",1409925,ead7c93a-ca70-4978-b47d-6d065724c374,ded84366-edd6-484e-9f23-8f7d27fd7f1a,"4,328,169.00",3655454,-1.00
4875,Love Hurts,2025,Love Hurts,"2,024.00",1226406,118b73f8-37fd-4c48-8569-67513685344c,a4535366-18c6-4bb4-8a17-12e3a9db05fa,"5,187,484.00",4338920,-1.00


Most albums have U\.S\. release years that coincide with or closely precede the associated film release year, with limited pre\-release offsets of up to three years\. Let's confirm the distribution in the next query

In [12]:
# Let's create a table of counts of albums released in the same year as the film, and in the year after the film
display(album_temporal_df['album_minus_film_release'].value_counts().sort_index())

# sum all album discrepancies less than -3
np.sum(album_temporal_df['album_minus_film_release'] < -3)

album_minus_film_release
-3.00       1
-1.00      52
0.00     1262
1.00      192
2.00        2
3.00        1
Name: count, dtype: int64

0

Findings: As you can see, a vast majority of the albums appear in the same year, or shortly before / after the film release date\. This means our film / album join data is pretty tight\.

### Data Integrity Exploration A2\.3: Inspect matches on the film long\-tail

In [13]:

cols = [
    "tmdb_id", "film_title", "film_year", "film_vote_count",
    "album_title", "album_us_release_year","rg_secondary_types",
    "soundtrack_type", "match_method", 
    "release_group_mbid", "release_mbid"
]
mask = (album_df["film_vote_count"] < 100) & (album_df["match_method"] == "title_contains_strict")

display(album_df.loc[mask, cols].head(50))

,tmdb_id,film_title,film_year,film_vote_count,album_title,album_us_release_year,rg_secondary_types,soundtrack_type,match_method,release_group_mbid,release_mbid
790,1284460,Alpha,2025,84,Alpha: Original Motion Picture Soundtrack,NaN,Soundtrack,songs,title_contains_strict,32970442-4dd9-4762-a688-f7fa9f7b36f8,9b602c60-fdf7-4d2b-b517-9f1690a8c84d
793,375199,Jai Gangaajal,2016,13,Jai Gangaajal,NaN,Soundtrack,unknown,title_contains_strict,9f2335b9-f8f8-47f2-a1c7-1382ccff40a4,c80613ed-f038-4a70-945d-9ac6a3114b33
794,1025881,Bobcat Moretti,2023,11,Bobcat Moretti (Original Motion Picture Soundt...,"2,024.00",Soundtrack,songs,title_contains_strict,7197799d-7e04-4be7-a716-8432c56dcb78,1cbb5e33-cfca-4311-85d4-e96378144ce8
796,613658,The Rescue,2020,59,The Rescue: Original Motion Picture Soundtrack,"2,021.00",Soundtrack,songs,title_contains_strict,20c4e5d1-55c9-49ef-82fe-c253408dd8c5,7c294a8b-46f8-44fa-9682-962074a262c8
799,466550,Drive,2019,59,Drive,NaN,Soundtrack,unknown,title_contains_strict,728c2dbf-0b60-41f3-8c46-d2f3d3343056,bab61726-6420-4069-9733-0ae29e033160
800,636548,Drive,2019,10,Drive,NaN,Soundtrack,unknown,title_contains_strict,728c2dbf-0b60-41f3-8c46-d2f3d3343056,bab61726-6420-4069-9733-0ae29e033160
801,454776,Fukrey Returns,2017,55,Fukrey Returns,NaN,Soundtrack,unknown,title_contains_strict,f9b7cdd9-fd8d-4ff2-b4f8-bfa56b6c029c,f9e01f9c-4611-4e0a-ab1e-b11241daab7e
802,479758,Luna,2018,38,Luna (Original Soundtrack),NaN,Soundtrack,songs,title_contains_strict,aa25109c-fe9b-43ce-9207-c69ffab610df,0a5113ea-cb9c-4a12-9d62-d20b1a0b21fb
804,876911,Parallel,2024,58,Parallel,NaN,Soundtrack,unknown,title_contains_strict,27038868-2e98-45b6-83bb-69b86af58e3e,848d227f-d8e0-420f-8456-183cbd437cf8
805,1148172,Do Patti,2024,57,Do Patti,NaN,Soundtrack,unknown,title_contains_strict,afadda1e-a2f7-4d49-937b-bc19342407f3,d7993c7e-39a8-48e4-b004-067997793ec0


Filtering to low–vote\-count films matched via title\_contains\_strict highlights a cluster of album associations with weaker supporting metadata, including frequent missing U\.S\. release years and a high prevalence of soundtrack\_type = unknown, often for generic or repeated titles \(e\.g\., Drive, Red, Luna\)\. 

These cases are not obviously incorrect, but the title\-only matching signal is relatively weak compared to IMDb\-exact matches or albums explicitly labeled as scores or soundtracks\. As a result, these rows are retained but flagged as lower\-confidence associations to be handled cautiously in downstream analysis rather than removed outright\.

### Data Integrity Exploration A2\.4: Non\-Latin Character Sets

In [14]:
import pandas as pd

# --- Helpers -----------------------------------------------------------------

def contains_any_in_ranges(s, ranges):
    """
    Return True if any character in s falls within any inclusive Unicode range
    in `ranges` (list of (start, end) codepoint tuples).
    """
    for ch in s:
        cp = ord(ch)
        for start, end in ranges:
            if start <= cp <= end:
                return True
    return False


# --- Ranges ------------------------------------------------------------------
# “Non-Latin” scripts we want to catch explicitly (then everything else non-latinish also folds in)
NON_LATIN_RANGES = [
    # CJK + Hangul
    (0x4E00, 0x9FFF),   # Han
    (0x3400, 0x4DBF),   # Han Ext A
    (0x3040, 0x309F),   # Hiragana
    (0x30A0, 0x30FF),   # Katakana
    (0xAC00, 0xD7AF),   # Hangul syllables
    (0x1100, 0x11FF),   # Hangul Jamo

    # Indic (fold into non_latin_script)
    (0x0900, 0x097F),   # Devanagari
    (0x0980, 0x09FF),   # Bengali
    (0x0A00, 0x0A7F),   # Gurmukhi
    (0x0A80, 0x0AFF),   # Gujarati
    (0x0B00, 0x0B7F),   # Oriya
    (0x0B80, 0x0BFF),   # Tamil
    (0x0C00, 0x0C7F),   # Telugu
    (0x0C80, 0x0CFF),   # Kannada
    (0x0D00, 0x0D7F),   # Malayalam

    # Thai
    (0x0E00, 0x0E7F),   # Thai
]

# Accented Latin letters
LATIN_EXT_RANGES = [
    (0x00C0, 0x00FF),  # Latin-1 Supplement
    (0x0100, 0x017F),  # Latin Extended-A
    (0x0180, 0x024F),  # Latin Extended-B
    (0x1E00, 0x1EFF),  # Latin Extended Additional
]

# “Latin-ish” extras that commonly appear in otherwise-Latin titles
# (smart quotes, non-breaking hyphens, roman numerals, ™/®,
# some fullwidth punctuation, etc.)
LATINISH_MISC_RANGES = [
    (0x0300, 0x036F),  # Combining diacritics
    (0x2000, 0x206F),  # General Punctuation (smart quotes, en-dash, etc.)
    (0x2100, 0x214F),  # Letterlike Symbols (™ etc.)
    (0x2150, 0x218F),  # Number Forms (Roman numerals like ⅩⅢ)
    (0x2460, 0x24FF),  # Enclosed Alphanumerics
    (0xFF00, 0xFFEF),  # Halfwidth/Fullwidth forms (includes ～)
    # NOTE: I intentionally removed (0x3000, 0x303F) because it includes 「」,
    # which are Japanese punctuation and you probably want those to count as non_latin_script.
]


def is_latinish_char(ch):
    """
    True if ch is ASCII OR falls into an allowed Latin/typographic block that we
    still consider 'Latin titles' for lookup purposes.
    """
    cp = ord(ch)

    # ASCII
    if cp < 128:
        return True

    # Accented Latin letters
    for start, end in LATIN_EXT_RANGES:
        if start <= cp <= end:
            return True

    # Typographic / symbol blocks we’re treating as still "Latin-ish"
    for start, end in LATINISH_MISC_RANGES:
        if start <= cp <= end:
            return True

    return False


def classify_title_script(text):
    """
    Buckets:
      - ascii_clean:        all characters are ASCII (best for naive lookups)
      - latin_typographic:  Latin text with diacritics and/or typographic punctuation/symbols
                            (usually fine, but normalize smart quotes/hyphens)
      - non_latin_script:   contains non-Latin scripts (Thai/CJK/Indic/etc) OR contains
                            other characters we don’t consider “latin-ish”
    """
    if pd.isna(text) or str(text).strip() == "":
        return "missing"

    s = str(text)

    # 1) Clean ASCII
    if all(ord(c) < 128 for c in s):
        return "ascii_clean"

    # 2) Known non-Latin scripts => higher risk bucket
    if contains_any_in_ranges(s, NON_LATIN_RANGES):
        return "non_latin_script"

    # 3) Everything else: if fully "latin-ish" => typographic bucket, otherwise non-Latin
    return "latin_typographic" if all(is_latinish_char(c) for c in s) else "non_latin_script"


# --- Prevalence (album-level, deduped) ---------------------------------------

album_level = album_df[["release_group_mbid", "album_title"]].drop_duplicates()

script_bucket = album_level["album_title"].apply(classify_title_script)

summary = (
    script_bucket.value_counts(dropna=False)
    .rename("album_count")
    .to_frame()
)
summary["pct_of_albums"] = (summary["album_count"] / summary["album_count"].sum() * 100).round(2)

display(summary)


,album_count,pct_of_albums
album_title,,
ascii_clean,4659,94.54
latin_typographic,196,3.98
non_latin_script,73,1.48


In [15]:
# Inspect examples for each bucket
for bucket in ["latin_typographic", "non_latin_script"]:
    print(f"\nExamples: {bucket}")
    display(album_level.loc[script_bucket == bucket].sample(10))



Examples: latin_typographic


,release_group_mbid,album_title
672,f6acd0b5-b22b-42d2-a61a-67f3b9164c79,Tchaikovsky’s Wife: Original Motion Picture So...
3647,0c9c713f-d841-49b9-8141-c74486bc6880,Les Mystères des majorettes
3258,160f13b8-17b6-40de-8bf2-f55ac5bf1a0e,"Heartbeat (From the “Lyle, Lyle, Crocodile” Or..."
2951,5892674d-d841-4ae5-8c02-1170abf481dd,Ant‐Man and the Wasp: Original Motion Picture ...
2793,dd937e7e-be00-425d-b96a-5588a03ca8af,Tomboy (From “The Assignment”)
413,68abe3ed-0768-426c-8a61-c918d163baaa,Yucatán
256,21406767-86ac-4104-804a-afd500068bde,The Hitman’s Bodyguard: Original Motion Pictur...
2923,e19b418c-8496-4b8a-85e7-3e6fed5506a6,Les fantômes d’Ismaël
3069,ccd78c22-08be-4fc1-a23d-1b93d636be81,"Cetto c’è, senzadubbiamente (Colonna sonora or..."
2888,66fa7114-0ea2-429e-bdd8-7f8c3508f99a,Braqueurs d’élite



Examples: non_latin_script


,release_group_mbid,album_title
4376,de655243-3975-4e8a-8b0f-a06fdec8a6b1,ReLIFE オリジナルサウンドトラック
891,9d3c5263-1aae-43c4-9bd6-8b538f2b69c3,映画「百日紅~Miss HOKUSAI~」オリジナル・サウンドトラック
3560,e05a87e9-f390-4000-956f-21303e32b4cd,银河系Disco
4029,b70133e7-0ba9-4f78-b1c0-dffdd844d367,真・女神転生Ⅲ NOCTURNE サウンドコレクション
5045,3a13c860-bd5f-4994-87e8-0c266e68040a,相棒-劇場版III- 巨大密室!特命系 絕命孤島 オリジナル･サウンドトラック
475,ce503a48-37b3-4026-a75e-d0da2a32a460,亡羊
1477,49daaf4f-352f-429f-8c61-4e4b3732098c,ONE PIECE FILM GOLD オリジナル・サウンドトラック
563,762940f3-6d70-4398-8895-cba2fff4d4a8,刺殺小說家
157,367d76f9-636d-4132-b9fe-39564be6528a,不说 (电影《从你的全世界路过》路过版主题曲)
2193,983ea8ab-624d-4cf2-ad55-76d9572782f9,New スーパーマリオブラザーズ


Findings: The vast majority of soundtrack albums in the dataset \(94\.5%\) have clean ASCII\-only titles, which represent the lowest\-friction case for downstream lookups and matching\. An additional ~4% fall into a latin\-typographic bucket—these are still fundamentally Latin titles, but include smart quotes, accented characters, or typographic symbols, and should be safe to retain with light normalization\. Only a small fraction of albums \(1\.5%\) contain non\-Latin scripts \(e\.g\., CJK, Indic, Thai\), which are more likely to introduce ambiguity or failure in external lookups; given their low prevalence, these can be flagged for caution or handled separately without materially affecting overall coverage\.

In [16]:
display(album_df[['album_title', 'release_title']])

,album_title,release_title
0,Fifty Shades of Grey: Original Motion Picture ...,Fifty Shades of Grey: Original Motion Picture ...
1,Fifty Shades of Grey: Original Motion Picture ...,Fifty Shades of Grey: Original Motion Picture ...
2,Cinderella,Cinderella
3,The Revenant: Original Motion Picture Soundtrack,The Revenant: Original Motion Picture Soundtrack
4,Star Wars: The Force Awakens: Original Motion ...,Star Wars: The Force Awakens: Original Motion ...
...,...,...
5204,Prassthanam,Prassthanam
5205,Panga,Panga
5206,Pets (Original Soundtrack),Pets (Original Soundtrack)
5207,The Book of Clarence: Original Motion Picture ...,The Book of Clarence: Original Motion Picture ...


### Data Integrity Exploration A2\.5: Reliability of tagging metadata for album genre

Question: How comprehensive is the album tagging data? Are we able to use them as a feature? Can we standardize their values?

In [17]:
tag_related_cols = ['release_group_mbid', 'rg_tags_text', 'release_tags_text', 
'label_tags_text']
albumtags_df = album_df[tag_related_cols]

# Count the nulls and percentage nulls in the tag-related columns
null_counts = albumtags_df.isna().sum()
null_pct = (albumtags_df.isna().mean() * 100).round(2)

null_summary = (
    pd.concat([null_counts, null_pct], axis=1)
      .rename(columns={0: "null_count", 1: "null_pct"})
)

display(null_summary)

display(albumtags_df.sample(50))

,null_count,null_pct
release_group_mbid,0,0.00
rg_tags_text,4209,80.80
release_tags_text,4804,92.22
label_tags_text,2281,43.79


,release_group_mbid,rg_tags_text,release_tags_text,label_tags_text
2324,e1397e4b-621f-496b-bc09-fe65c969fef3,NaN,NaN,NaN
5008,dde478aa-05de-406b-b2d2-adc8740bd4f1,NaN,NaN,NaN
4241,c3956c2f-794c-4f47-b748-f2e5f3508962,classical,NaN,clean up | en.wiki != ko.wiki
3611,634724e2-4225-384f-a6c9-533e8b7e43e5,NaN,NaN,NaN
2234,f8eb99af-b443-4461-8448-db7a847a77d6,NaN,NaN,indian | filmi | indian pop
5206,7f4d1537-5670-4b8e-a68b-aebe72de0370,NaN,NaN,score | soundtrack | soundtrack label | classi...
2382,bd4bef98-7b49-4414-bc60-0bd608b231fb,NaN,NaN,soundtrack | soundtrack label
2841,89f8d348-818d-4a5e-a708-9974a4a39d33,electronic | hip hop | rock,NaN,NaN
2969,6853c446-912c-4d99-b5b1-b5b0a6ad437d,NaN,NaN,soundtrack | soundtrack label
3365,84aa00b9-c920-4d5b-9dff-33c737a37299,electronic | experimental | instrumental | jazz,NaN,british | electronic | rock | uk | 21 | 80s | ...


Finding: Album\-level genre tags from MusicBrainz are sparsely populated \(≈19% coverage\) and inconsistently applied across the soundtrack corpus\. Release\-level tags are even more sparse, while label\-level tags exhibit highly unpredictable and non\-genre behavior based on visual inspection \(e\.g\., organizational metadata, locations, and free\-form noise\)\. As a result, genre\-based analysis will be limited to exploratory, conditional subsets of albums where album\-level tags are present, rather than treated as a complete or reliable feature across the full dataset\.

Let's now locate the canonical values that we can use for album\-level genre\.

In [18]:
# 1. Pull out the column and keep only non-nulls
tags_series = album_df['rg_tags_text'].dropna().str.strip()

# print(len(tags_series), "\n", tags_series)

# 2. Explode the tags based on the | delimiter. Specify regex = False so | is not treated like an OR
tags_exploded = tags_series.str.split(" | ", regex = False).explode()
# print(len(tags_exploded), "\n", tags_exploded)

# 3. Count all the values
tag_counts_df = tags_exploded.value_counts().reset_index(name='count')
display(tag_counts_df)
tag_counts_df.to_csv("rg_tags_value_counts.csv", index = False)

,rg_tags_text,count
0,soundtrack,253
1,electronic,245
2,classical,241
3,pop,193
4,rock,137
...,...,...
349,james bond 007,1
350,freestyle,1
351,cool jazz,1
352,character: indigo zap,1


We investigate the community\-specified values\. We don't need redundant labels like "This is a soundtrack" or TV/Video game theme\. 

In a later notebook, we'll group genres into a handful of buckets and create boolean flags for them\. For now, let's keep this notebook focused on inspecting the data

# QA of the Track Table

## Basic Track\-level Checks

In [19]:
# Read the tracks table CSV

tracks_df = pd.read_csv("./pipeline/2.2.MUSICBRAINZ_mv_tmdb_soundtrack_track_spine_2015_2025.csv")
                         
display(tracks_df.head())
display(tracks_df.columns)

,tmdb_id,film_title,release_group_id,release_id,match_method,album_us_release_date,us_date_has_missing_month,us_date_has_missing_day,medium_id,disc_number,...,recording_artist_credit,recording_first_release_year,recording_first_release_month,recording_first_release_day,isrcs_text,recording_tags_text,work_ids_text,work_titles_text,composer_names_text,lyricist_names_text
0,158852,Tomorrowland,1527294,1608979,imdb_exact,2015-06-09,False,False,1692109,1,...,Michael Giacchino,NaN,NaN,NaN,USWD11570549,NaN,14179222,Frank Frank,Michael Giacchino,NaN
1,664425,Alone,4404592,5269969,title_contains_strict,NaN,NaN,NaN,5726478,1,...,Frederik Wiedmann,NaN,NaN,NaN,NaN,NaN,14876176,Isolate,Frederik Wiedmann,NaN
2,675799,Alone,4404592,5269969,title_contains_strict,NaN,NaN,NaN,5726478,1,...,Frederik Wiedmann,NaN,NaN,NaN,NaN,NaN,14876176,Isolate,Frederik Wiedmann,NaN
3,661950,Alone,4404592,5269969,title_contains_strict,NaN,NaN,NaN,5726478,1,...,Frederik Wiedmann,NaN,NaN,NaN,NaN,NaN,14876176,Isolate,Frederik Wiedmann,NaN
4,664425,Alone,4404592,5269969,title_contains_strict,NaN,NaN,NaN,5726478,1,...,Frederik Wiedmann,NaN,NaN,NaN,NaN,NaN,14876184,Screamers,Frederik Wiedmann,NaN


Index(['tmdb_id', 'film_title', 'release_group_id', 'release_id',
       'match_method', 'album_us_release_date', 'us_date_has_missing_month',
       'us_date_has_missing_day', 'medium_id', 'disc_number', 'medium_format',
       'track_id', 'track_number', 'track_title', 'track_length_ms',
       'recording_id', 'recording_mbid', 'recording_title',
       'recording_length_ms', 'recording_artist_credit',
       'recording_first_release_year', 'recording_first_release_month',
       'recording_first_release_day', 'isrcs_text', 'recording_tags_text',
       'work_ids_text', 'work_titles_text', 'composer_names_text',
       'lyricist_names_text'],
      dtype='object')

In [20]:
print("Rows:", len(tracks_df))
print("Unique track_ids:", tracks_df["track_id"].nunique())
print("Unique recording_ids:", tracks_df["recording_id"].nunique())

Rows: 85842
Unique track_ids: 81565
Unique recording_ids: 81107


### Data Integrity Exploration T1\.1: Describe ratios of relationships

Question: Multiple tracks are associated with albums\. Tracks can have 1 or more recording\. Mediums \(LP, CDs, \.\.\.\) have multiple tracks\. What are the averages of each?

In [21]:
# Quick ratios (helpful smell tests)
if all(c in tracks_df.columns for c in ["tmdb_id", "track_id", "recording_mbid", "medium_id"]):
    films = tracks_df["tmdb_id"].nunique(dropna=True)
    albums = tracks_df["release_group_id"].nunique(dropna=True)
    tracks = tracks_df["track_id"].nunique(dropna=True)
    recs  = tracks_df["recording_mbid"].nunique(dropna=True)
    meds  = tracks_df["medium_id"].nunique(dropna=True)
    print("\nRATIOS (unique-based)")
    print(f"  tracks_per_film (unique track_id / unique tmdb_id): {tracks/films:,.2f}")
    print(f"  tracks_per_album (unique track_id / unique release_group_id): {tracks/albums:,.2f}")
    print(f"  recordings_per_track (unique recording_mbid / unique track_id): {recs/tracks:,.3f}")
    print(f"  tracks_per_medium (unique track_id / unique medium_id): {tracks/meds:,.2f}")


RATIOS (unique-based)
  tracks_per_film (unique track_id / unique tmdb_id): 17.14
  tracks_per_album (unique track_id / unique release_group_id): 16.61
  recordings_per_track (unique recording_mbid / unique track_id): 0.994
  tracks_per_medium (unique track_id / unique medium_id): 16.19


Findings: In reality, we are probably only going to make use of tracks\_per\_film or tracks\_per\_album, so those are the only interesting ones we can use for estimating\.

### Data Integrity Exploration T1\.2: Key completeness and uniqueness

Question: How well populated are our different keys?

In [22]:
# T1.2 Basic uniqueness + nulls for key IDs
KEY_IDS = ["tmdb_id", "release_group_id", "release_id", "medium_id", "track_id", "recording_mbid"]

out = []
for c in KEY_IDS:
    dup_ct = int(tracks_df.duplicated(subset = [c]).sum())
    out.append({
        "col": c,
        "n_unique": tracks_df[c].nunique(dropna=True),
        "null_ct": int(tracks_df[c].isna().sum()),
        "null_pct": float(tracks_df[c].isna().mean() * 100),
        "duplicated_rows": dup_ct,
        "duplicated_pct": float(dup_ct/len(tracks_df) * 100)
    })

pd.DataFrame(out).sort_values("col")


,col,n_unique,null_ct,null_pct,duplicated_rows,duplicated_pct
3,medium_id,5038,0,0.00,80804,94.13
5,recording_mbid,81107,0,0.00,4735,5.52
1,release_group_id,4911,0,0.00,80931,94.28
2,release_id,4911,0,0.00,80931,94.28
0,tmdb_id,4760,0,0.00,81082,94.45
4,track_id,81565,0,0.00,4277,4.98


Findings: All IDs have been well\-populated by our SQL scripts\! However, there are about 4277 tracks that are duplicated \-\- we should investigate\.

In [23]:
# Isolate the duplicated tracks
dup_mask = tracks_df['track_id'].duplicated(keep = False)
dup_tracks = tracks_df[dup_mask].copy()

print("Unique duplicated track_ids:", dup_tracks['track_id'].nunique())

# Join duplicated track rows to albums_df (use release_group_id)
# keep album columns distinct
joined = dup_tracks.merge(
    album_df,
    on = ['release_group_id'],
    how="left",
    suffixes=("", "_album"),
    validate="m:m"
)

print("dup_tracks rows:", len(dup_tracks))
print("joined rows:   ", len(joined))
joined


Unique duplicated track_ids: 3352
dup_tracks rows: 7629
joined rows:    18973


,tmdb_id,film_title,release_group_id,release_id,match_method,album_us_release_date,us_date_has_missing_month,us_date_has_missing_day,medium_id,disc_number,...,release_tags_text,label_names,label_mbids,label_tags_text,rg_rating,rg_rating_count,release_meta_date_added,release_meta_info_url,release_meta_amazon_asin,release_meta_cover_art_presence
0,664425,Alone,4404592,5269969,title_contains_strict,NaN,NaN,NaN,5726478,1,...,NaN,KaleidoSound,29aa4859-81a4-467f-b542-2e418884ac18,NaN,NaN,NaN,2025-10-14 17:32:03.872 -0400,NaN,NaN,present
1,664425,Alone,4404592,5269969,title_contains_strict,NaN,NaN,NaN,5726478,1,...,NaN,KaleidoSound,29aa4859-81a4-467f-b542-2e418884ac18,NaN,NaN,NaN,2025-10-14 17:32:03.872 -0400,NaN,NaN,present
2,664425,Alone,4404592,5269969,title_contains_strict,NaN,NaN,NaN,5726478,1,...,NaN,KaleidoSound,29aa4859-81a4-467f-b542-2e418884ac18,NaN,NaN,NaN,2025-10-14 17:32:03.872 -0400,NaN,NaN,present
3,675799,Alone,4404592,5269969,title_contains_strict,NaN,NaN,NaN,5726478,1,...,NaN,KaleidoSound,29aa4859-81a4-467f-b542-2e418884ac18,NaN,NaN,NaN,2025-10-14 17:32:03.872 -0400,NaN,NaN,present
4,675799,Alone,4404592,5269969,title_contains_strict,NaN,NaN,NaN,5726478,1,...,NaN,KaleidoSound,29aa4859-81a4-467f-b542-2e418884ac18,NaN,NaN,NaN,2025-10-14 17:32:03.872 -0400,NaN,NaN,present
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18968,318941,Darling,2334586,2638371,title_contains_strict,NaN,NaN,NaN,2864921,1,...,NaN,NaN,NaN,NaN,NaN,NaN,2020-03-10 15:09:37.817 -0400,NaN,NaN,present
18969,1281775,FLOW,3912825,4653080,title_contains_strict,NaN,NaN,NaN,5063579,1,...,NaN,Milan,41099e29-7301-4fa3-86a4-8d510157b275,burbank | california | clean up | score | soun...,NaN,NaN,2024-11-12 05:42:43.175 -0500,NaN,NaN,present
18970,1281775,FLOW,3912825,4653080,title_contains_strict,NaN,NaN,NaN,5063579,1,...,NaN,Milan,41099e29-7301-4fa3-86a4-8d510157b275,burbank | california | clean up | score | soun...,NaN,NaN,2024-11-12 05:42:43.175 -0500,NaN,NaN,present
18971,823219,Flow,3912825,4653080,title_contains_strict,NaN,NaN,NaN,5063579,1,...,NaN,Milan,41099e29-7301-4fa3-86a4-8d510157b275,burbank | california | clean up | score | soun...,NaN,NaN,2024-11-12 05:42:43.175 -0500,NaN,NaN,present


Findings: We need to do a track cleanup at a later stage after we do our album cleanup

### Data Integrity Exploration T1\.3: Track number analysis

Question: What is the statistical characteristics of the track\_number column?

In [24]:
# T1.3 Track_number sanity (nulls, min/max, and weird values)

s = pd.to_numeric(tracks_df["track_number"], errors="coerce")

print("null_ct:", int(s.isna().sum()))
print("median:", float(s.median()))
print("min:", float(s.min()), "max:", float(s.max()))
print("<=0 ct:", int((s <= 0).sum()))
print(">=100 ct:", int((s >= 100).sum()))
display(s.describe(percentiles=[0.5, 0.9, 0.95, 0.99]))


null_ct: 0
median: 10.0
min: 0.0 max: 107.0
<=0 ct: 3
>=100 ct: 8


count   85,842.00
mean        11.49
std          8.83
min          0.00
50%         10.00
90%         23.00
95%         27.00
99%         39.00
max        107.00
Name: track_number, dtype: float64

Findings: Track numbering appears well\-behaved across the dataset\. All tracks have a populated track\_number, with a median of 10 and a 95th percentile of 27, consistent with typical soundtrack album structures\. The long tail is limited \(99th percentile = 39\), indicating no evidence of track\-level join inflation or structural duplication\. A very small number of edge cases were observed \(3 tracks numbered 0 and 8 tracks ≥100\), which likely reflect MusicBrainz metadata quirks or non\-canonical releases rather than systemic issues\. No corrective action required at this stage\.

### Data Integrity Exploration T1\.4: Track length basic stats

Question: Does the track length data look regular?

In [25]:
# T1.4 length fields basic stats (track_length_ms + recording_length_ms)
for c in ["track_length_ms", "recording_length_ms"]:

    s = pd.to_numeric(tracks_df[c], errors="coerce")
    print("\n", c)
    print("null_ct:", int(s.isna().sum()))
    print("Median", s.median())
    print("zero_ct:", int((s == 0).sum()))
    print("neg_ct:", int((s < 0).sum()))
    display(s.describe(percentiles=[0.5, 0.9, 0.95, 0.99]))



 track_length_ms
null_ct: 2549
Median 141000.0
zero_ct: 0
neg_ct: 0


count      83,293.00
mean      157,429.33
std        97,104.35
min         1,000.00
50%       141,000.00
90%       270,000.00
95%       317,000.00
99%       457,004.80
max     4,673,773.00
Name: track_length_ms, dtype: float64


 recording_length_ms
null_ct: 1978
Median 141000.0
zero_ct: 0
neg_ct: 0


count      83,864.00
mean      157,327.95
std        99,546.95
min         1,000.00
50%       141,000.00
90%       269,621.50
95%       317,000.00
99%       457,000.00
max     6,540,000.00
Name: recording_length_ms, dtype: float64

Findings: Track\- and recording\-level duration fields exhibit strong internal consistency\. Both track\_length\_ms and recording\_length\_ms share an identical median \(141 seconds\) and nearly identical upper percentiles, indicating correct alignment between track\- and recording\-level joins and no evidence of track duplication or inflation\. The observed duration ranges are consistent with expected soundtrack cue lengths\. A small number of extreme long\-duration recordings were observed, likely reflecting full\-score or suite\-style releases, but these do not materially affect the overall distribution\. 

Given the high degree of overlap and consistency between the two fields, track and recording durations are effectively redundant for downstream analysis, and a single duration attribute can be selected without loss of information\.

### Data Integrity Exploration T1\.5: Text field check

Question: Which of the text fields are reliably populated?

In [26]:
# %% [13] basic text-field empties (treat "" as missing)
TEXT_COLS = [
    "track_title",
    "recording_title",
    "recording_artist_credit",
    "composer_names_text",
    "lyricist_names_text",
    "isrcs_text",
    "recording_tags_text",
]

out = []
for c in TEXT_COLS:
    s = tracks_df[c]
    empty = s.isna() | (s.astype(str).str.strip() == "")
    out.append({
        "col": c,
        "empty_ct": int(empty.sum()),
        "empty_pct": float(empty.mean() * 100)
    })

pd.DataFrame(out).sort_values("empty_pct", ascending=False)


,col,empty_ct,empty_pct
6,recording_tags_text,82764,96.41
4,lyricist_names_text,82483,96.09
3,composer_names_text,62272,72.54
5,isrcs_text,44616,51.97
0,track_title,0,0.00
1,recording_title,0,0.00
2,recording_artist_credit,0,0.00


Findings: Text\-based metadata fields exhibit highly uneven coverage across recordings\. Core identification attributes such as track\_title, recording\_title, and recording\_artist\_credit are fully populated, indicating strong integrity for essential track and artist identification\. In contrast, several enrichment fields show substantial sparsity: composer names are missing for over 70% of rows, lyricist names for over 96%, and recording tags for over 96%\. ISRC coverage is moderate, with approximately half of recordings lacking identifiers\. This pattern is consistent with MusicBrainz’ strength in core cataloging metadata rather than comprehensive credit or enrichment data for soundtrack recordings\.

From a previous test on album\_df, it looks like 100% of the films have relevant composer data, so we should ignore track\-level composer data in favor of TMDB's\.

In [27]:
# %% [14] quick "what values are we dealing with" for medium_format + match_method
for c in ["medium_format", "match_method"]:
    if c not in tracks_df.columns:
        print(f"{c}: missing")
        continue
    print("\n", c)
    display(tracks_df[c].value_counts(dropna=False).head(30))



 medium_format


medium_format
Digital Media    70678
CD               11963
NaN               1502
12" Vinyl          617
CD-R               315
HQCD               277
Vinyl              277
10" Vinyl           45
Data CD             42
SHM-CD              29
SACD                28
8cm CD+G            21
Cassette            20
Blu-spec CD         19
7" Vinyl             4
Mixed Mode CD        2
DVD-Video            1
DVD-Audio            1
Piano Roll           1
Name: count, dtype: int64


 match_method


match_method
title_contains_strict    56895
imdb_exact               28947
Name: count, dtype: int64

Findings: In the current pipeline, a single canonical release\_id \(the first one created in MusicBrainz\) is deterministically selected per release\_group, and only the tracks associated with that release are enumerated\. While the chosen release may reflect a physical format \(e\.g\., CD or vinyl\) based on MusicBrainz’s internal creation order, this does not imply that a digital release does not exist for the album\. Critically, tracklists across digital and physical releases within the same release group are typically identical at the recording level\. As a result, medium format selection does it materially affect downstream Last\.fm lookups, which operate on song identity rather than physical release packaging\.

## Association checks with Track

### T2\.1 Albums with extreme numbers of tracks

Question: Can we identify the albums with an unusual number of tracks?

In [28]:
# Films with extreme track counts
tpf = tracks_df.groupby("tmdb_id")["track_id"].nunique(dropna=True).rename("tracks_per_film")

top = tpf.sort_values(ascending=False).head(20).to_frame()

bottom = tpf.sort_values(ascending=True).head(20).to_frame()

bottom = bottom.join(tracks_df.groupby("tmdb_id")["film_title"].first(), how="left")

print("TOP 20")
display(top)


TOP 20


,tracks_per_film
tmdb_id,
346757,592
992190,357
1362052,319
866741,208
507089,130
762509,124
420817,111
643550,107
1062372,103


In [29]:
print("BOTTOM 20")
display(bottom)

BOTTOM 20


,tracks_per_film,film_title
tmdb_id,,
1024721,1,Monolith
585973,1,Destiny
587130,1,The Reckoning
588656,1,El verano que vivimos
996067,1,NightMare
591662,1,No Looking Back
324578,1,Butterfly
983311,1,Eye for an Eye
981314,1,Alien Invasion


Finding:  Track counts per film show a wide but interpretable range\. Most films fall within an expected soundtrack size, while a small number of films exhibit unusually high track counts \(e\.g\., 70–130\+ tracks\), likely reflecting extended scores, deluxe editions, or multi\-disc soundtrack releases rather than structural duplication\. At the opposite extreme, a subset of films that I have never heard of is associated with only a single track, which is consistent with films that have a minimal soundtrack presence or only a single identified theme or song in MusicBrainz\. Fortunately, these extremes are limited in number and do not indicate a systematic join inflation, but they highlight edge cases that warrant awareness in downstream analysis\.

# QA of the Wide Table

In [30]:
# Read the wide table CSV
wide_df = pd.read_csv("./pipeline/2.2.MUSICBRAINZ_mv_tmdb_soundtrack_wide_track_2015_2025.csv")

display(wide_df.columns)
display(wide_df.head())


/tmp/ipykernel_112/972477774.py:2: DtypeWarning: Columns (45,58,84) have mixed types. Specify dtype option on import or set low_memory=False.
  wide_df = pd.read_csv("./pipeline/2.2.MUSICBRAINZ_mv_tmdb_soundtrack_wide_track_2015_2025.csv")


Index(['tmdb_id', 'film_title', 'film_adult', 'film_runtime_min',
       'film_genres', 'film_rating', 'film_vote_count', 'film_mpaa_rating',
       'film_original_title', 'film_language_name', 'film_imdb_id',
       'film_wikidata_id', 'film_countries', 'film_year', 'film_release_date',
       'film_popularity', 'film_budget', 'film_revenue', 'film_studios',
       'film_director', 'film_soundtrack_composer_raw', 'film_top_cast',
       'film_keywords', 'film_ingested_at', 'release_group_id',
       'release_group_mbid', 'release_id', 'release_mbid', 'album_title',
       'rg_primary_type', 'rg_secondary_types', 'match_method',
       'album_soundtrack_type', 'album_notes', 'album_matched_at',
       'album_us_release_date', 'album_us_release_year',
       'album_us_release_month_min_observed',
       'album_us_release_day_min_observed', 'us_date_has_missing_month',
       'us_date_has_missing_day', 'us_any_event_missing_month',
       'us_any_event_missing_day', 'release_title', 'rel

,tmdb_id,film_title,film_adult,film_runtime_min,film_genres,film_rating,film_vote_count,film_mpaa_rating,film_original_title,film_language_name,...,recording_artist_credit,isrcs_text,recording_first_release_year,recording_first_release_month,recording_first_release_day,recording_tags_text,work_ids_text,work_titles_text,composer_names_text,lyricist_names_text
0,1027160,Alone in the Dark,False,90,"Horror, Thriller",4.77,13,NR,Alone in the Dark,English,...,Philippe Vachey,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1027160,Alone in the Dark,False,90,"Horror, Thriller",4.77,13,NR,Alone in the Dark,English,...,Philippe Vachey,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,322855,Shakedown,False,72,Documentary,5.04,14,NR,Shakedown,English,...,Buc Fifty,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,322855,Shakedown,False,72,Documentary,5.04,14,NR,Shakedown,English,...,Rob the Viking,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,322855,Shakedown,False,72,Documentary,5.04,14,NR,Shakedown,English,...,Son Doobie,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Basic integrity checks

### Data Integrity Exploration W1\.1 Row count reconciliation

Question: Did we construct the wide table correctly?

In [31]:
# W1.1 shape + row-count reconciliation (wide vs album vs track)
print("wide rows:", len(wide_df))
print("wide cols:", wide_df.shape[1])

print("album rows:", len(album_df))
print("track rows:", len(tracks_df))

print("wide / tracks ratio:", (len(wide_df) / len(tracks_df)) if len(tracks_df) else np.nan)


wide rows: 85842
wide cols: 88
album rows: 5209
track rows: 85842
wide / tracks ratio: 1.0


Finding: Yes, the wide table should have a row count equal to its most granular entity, which is the track\.

### Data Integrity Exploration W1\.2: Uniqueness of TMDB\_ID and Track\_ID

Question: Are there duplicate tmdb\_id and track\_id rows?

In [32]:
# W1.2 grain check: uniqueness at intended key (tmdb_id, track_id)
KEY = ["tmdb_id", "track_id"]

dup_rows = wide_df.duplicated(subset=KEY).sum()
print("duplicate (tmdb_id, track_id) rows:", int(dup_rows))
print("duplicate pct:", float(dup_rows / len(wide_df) * 100) if len(wide_df) else np.nan)

if dup_rows:
    display(
        wide_df.loc[wide_df.duplicated(subset=KEY, keep=False), KEY]
              .value_counts()
              .head(25)
              .rename("row_ct")
              .reset_index()
    )


duplicate (tmdb_id, track_id) rows: 0
duplicate pct: 0.0


In [33]:
# grain check: uniqueness at (release_group_id, track_id)
KEY = ["release_group_id", "track_id"]

dup_rows = wide_df.duplicated(subset=KEY).sum()
print("duplicate (release_group_id, track_id) rows:", int(dup_rows))
print("duplicate pct:", float(dup_rows / len(wide_df) * 100) if len(wide_df) else np.nan)

if dup_rows:
    display(
        wide_df.loc[wide_df.duplicated(subset=KEY, keep=False), KEY]
              .value_counts()
              .head(25)
              .rename("row_ct")
              .reset_index()
    )


duplicate (release_group_id, track_id) rows: 4277
duplicate pct: 4.982409543114094


,release_group_id,track_id,row_ct
0,3611817,45893104,6
1,3611817,45893093,6
2,3611817,45893110,6
3,3611817,45893102,6
4,3611817,45893101,6
5,3611817,45893100,6
6,3611817,45893099,6
7,3611817,45893098,6
8,3611817,45893097,6
9,3611817,45893096,6


Findings: This confirms that some cleanup is still required, but the issue is well\-scoped\. The Wide table is clean at the intended analytical grain—each \(tmdb\_id, track\_id\) pair is unique—indicating that the joins themselves did not introduce duplication\. However, approximately 5% of rows share the same \(release\_group\_id, track\_id\) combination\. This reflects cases where a single soundtrack album is associated with multiple films, typically due to title collisions, remakes, or ambiguous soundtrack matching upstream\. In other words, the same album’s tracks are being reused across films, not duplicated within a film\. This validates the need for a post\-matching canonicalization step at the film–album level, while also confirming that the Wide table construction is otherwise sound\.

### Data Integrity Exploration W1\.3: 

Question: Do the ids between Track and Wide match?

In [34]:
# W1.3 join integrity: wide keys vs track spine keys (did we invent or drop rows?)
w_keys = wide_df[["tmdb_id", "track_id"]].drop_duplicates()
t_keys = tracks_df[["tmdb_id", "track_id"]].drop_duplicates()

w_only = w_keys.merge(t_keys, on=["tmdb_id", "track_id"], how="left", indicator=True).query("_merge == 'left_only'")
t_only = t_keys.merge(w_keys, on=["tmdb_id", "track_id"], how="left", indicator=True).query("_merge == 'left_only'")

print("unique (tmdb_id, track_id) in wide:", len(w_keys))
print("unique (tmdb_id, track_id) in tracks:", len(t_keys))
print("wide keys not in tracks:", len(w_only))
print("tracks keys not in wide:", len(t_only))

if len(w_only):
    print("\nSample wide-only keys:")
    display(w_only.head(10))

if len(t_only):
    print("\nSample tracks-only keys:")
    display(t_only.head(10))


unique (tmdb_id, track_id) in wide: 85842
unique (tmdb_id, track_id) in tracks: 85842
wide keys not in tracks: 0
tracks keys not in wide: 0


Findings: The Wide table preserves the track spine exactly\. The number of unique \(tmdb\_id, track\_id\) pairs is identical in both the track table and the Wide table, with no keys added or dropped during the join\. This confirms that the Wide table construction did not introduce any row inflation or data loss and is a faithful projection of the underlying track spine\.

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=b124131d-2024-4253-bb46-8043aed4b78f' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>